# Master Evaluation: Robustness & Metrics Extraction

This notebook serves as the primary computation engine for the robustness evaluation. It applies five different adversarial attacks (FGSM, PGD, DeepFool, C&W, and Targeted I-FGSM) across three pre-trained models (MobileNetV2, EfficientNetB0, InceptionV3) using a subset of 100 images from MiniImageNet.

**Goal:** Calculate the retained Accuracy, Attack Success Rate (ASR), and average $L_2$ distortion for each model-attack combination.
**Output:** A `robustness_metrics.csv` file that will be used by subsequent visualization notebooks (Radar Charts, Heatmaps, etc.) to quickly plot the results without re-computing the attacks.

*Note: This script performs heavy computations and may take 1-2 hours to complete on a standard GPU.*

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import glob

# Import models
from tensorflow.keras.applications import EfficientNetB0, InceptionV3, MobileNetV2

# Import preprocessing and decoding
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inc
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob

In [ ]:
# Dictionary with model configurations
models_dict = {
    'MobileNetV2': {
        'model': MobileNetV2(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_mob,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0,
        'box_min': -1.0,
        'box_max': 1.0
    },
    'EfficientNetB0': {
        'model': EfficientNetB0(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_eff,
        'clip_min': 0.0,
        'clip_max': 255.0,
        'eps_scale': 127.5,
        'box_min': 0.0,
        'box_max': 255.0
    },
    'InceptionV3': {
        'model': InceptionV3(weights='imagenet'),
        'target_size': (299, 299),
        'preprocess_fn': preprocess_inc,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0,
        'box_min': -1.0,
        'box_max': 1.0
    }
}

# Freeze weights as we only need to perform inference and compute gradients w.r.t inputs
for config in models_dict.values():
    config['model'].trainable = False

### Helper Functions & Data Loading

In [ ]:
def load_and_preprocess_image(img_path, target_size, preprocess_fn):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = preprocess_fn(img) # Dynamic preprocessing
    img = tf.expand_dims(img, axis=0) # Add batch dimension
    return img

In [ ]:
# Define the path to the 100 images folder
dataset_path = '../images/miniimagenet_random_100/'
image_paths = glob.glob(os.path.join(dataset_path, '*.*'))

# Limit to 100 just in case
image_paths = image_paths[:100]
print(f"Total images loaded for the experiment: {len(image_paths)}")

### Attack Implementations
We define the 5 attacks here. We use standardized hyperparameters to ensure a fair comparison across models.

In [ ]:
loss_object = tf.keras.losses.CategoricalCrossentropy()

In [ ]:
def fgsm_attack(img, label, epsilon, model, clip_min, clip_max):
    with tf.GradientTape() as tape:
        tape.watch(img)
        pred = model(img, training=False)
        loss = loss_object(label, pred)
    grad = tape.gradient(loss, img)
    adv_img = img + epsilon * tf.sign(grad)
    return tf.clip_by_value(adv_img, clip_min, clip_max)

In [ ]:
def pgd_attack(img, label, epsilon, model, clip_min, clip_max, iters=10):
    alpha = epsilon / (iters / 2.0)
    adv_img = tf.identity(img)
    for _ in range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_img)
            pred = model(adv_img, training=False)
            loss = loss_object(label, pred)
        grad = tape.gradient(loss, adv_img)
        adv_img = adv_img + alpha * tf.sign(grad)
        perturbation = tf.clip_by_value(adv_img - img, -epsilon, epsilon)
        adv_img = tf.clip_by_value(img + perturbation, clip_min, clip_max)
    return adv_img

In [ ]:
def cw_attack(img, label, model, box_min, box_max, c_weight=1.0, max_iters=100):
    modifier = tf.Variable(tf.zeros_like(img))
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)
    
    for _ in range(max_iters):
        with tf.GradientTape() as tape:
            adv_norm = 0.5 * (tf.tanh(modifier) + 1.0)
            adv_img = adv_norm * (box_max - box_min) + box_min
            l2_loss = tf.reduce_sum(tf.square(adv_img - img))
            preds = model(adv_img, training=False)
            real_prob = tf.reduce_sum(label * preds, axis=1)
            other_prob = tf.reduce_max((1.0 - label) * preds, axis=1)
            f_loss = tf.maximum(0.0, real_prob - other_prob)
            total_loss = l2_loss + c_weight * f_loss
        grads = tape.gradient(total_loss, [modifier])
        optimizer.apply_gradients(zip(grads, [modifier]))
        
    adv_norm = 0.5 * (tf.tanh(modifier) + 1.0)
    return adv_norm * (box_max - box_min) + box_min

In [ ]:
def deepfool_attack(img, model, clip_min, clip_max, num_classes=10, overshoot=0.02, max_iter=20):
    adv_img = tf.identity(img)
    top_classes = tf.argsort(model(adv_img)[0], direction='DESCENDING')[:num_classes]
    orig_label = tf.cast(top_classes[0], tf.int32)
    curr_label = orig_label
    iteration = 0
    
    while curr_label == orig_label and iteration < max_iter:
        perturbation = tf.zeros_like(adv_img)
        min_w_norm = float('inf')
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(adv_img)
            preds = model(adv_img, training=False)
            orig_loss = preds[0, orig_label]
        orig_grad = tape.gradient(orig_loss, adv_img)
        
        for k in range(1, num_classes):
            target_class = tf.cast(top_classes[k], tf.int32)
            with tf.GradientTape() as tape_k:
                tape_k.watch(adv_img)
                target_loss = model(adv_img, training=False)[0, target_class]
            target_grad = tape_k.gradient(target_loss, adv_img)
            w_k = target_grad - orig_grad
            f_k = target_loss - orig_loss
            w_norm = tf.norm(w_k)
            if w_norm == 0: continue
            distance = tf.abs(f_k) / (w_norm + 1e-6)
            if distance < min_w_norm:
                min_w_norm = distance
                perturbation = (distance * w_k) / (w_norm + 1e-6)
                
        adv_img = tf.clip_by_value(adv_img + (1 + overshoot) * perturbation, clip_min, clip_max)
        curr_label = tf.cast(tf.argmax(model(adv_img, training=False)[0]), tf.int32)
        iteration += 1
    return adv_img

In [ ]:
def targeted_ifgsm_attack(img, target_label, epsilon, model, clip_min, clip_max, iters=20):
    alpha = epsilon / (iters / 2.0)
    adv_img = tf.identity(img)
    for _ in range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_img)
            pred = model(adv_img, training=False)
            loss = loss_object(target_label, pred)
        grad = tape.gradient(loss, adv_img)
        adv_img = adv_img - alpha * tf.sign(grad)
        perturbation = tf.clip_by_value(adv_img - img, -epsilon, epsilon)
        adv_img = tf.clip_by_value(img + perturbation, clip_min, clip_max)
    return adv_img

### Global Evaluation Loop
We evaluate each model against all attacks. We assume a baseline $\varepsilon=0.05$ (scaled accordingly) and a target class `301` (Ladybug) for the targeted attack.

In [ ]:
# Standardized Hyperparameters
BASE_EPSILON = 0.05
TARGET_CLASS = 301

results = []

for model_name, config in models_dict.items():
    print(f"\n{'='*80}")
    print(f"EVALUATING MODEL: {model_name}")
    print(f"{'='*80}\n")
    
    model = config['model']
    eps = BASE_EPSILON * config['eps_scale']
    c_min, c_max = config['clip_min'], config['clip_max']
    
    # Initialize trackers for this model
    metrics = {
        'Baseline': {'correct': 0, 'l2': 0},
        'FGSM':     {'correct': 0, 'success': 0, 'l2': 0},
        'PGD':      {'correct': 0, 'success': 0, 'l2': 0},
        'C&W':      {'correct': 0, 'success': 0, 'l2': 0},
        'DeepFool': {'correct': 0, 'success': 0, 'l2': 0},
        'T-IFGSM':  {'correct': 0, 'success': 0, 'l2': 0}
    }
    
    for i, img_path in enumerate(image_paths):
        if (i+1) % 10 == 0:
            print(f"Processing image {i+1}/100...")
            
        img = load_and_preprocess_image(img_path, config['target_size'], config['preprocess_fn'])
        
        # Baseline
        orig_preds = model(img, training=False)
        orig_idx = tf.argmax(orig_preds, axis=-1).numpy()[0]
        one_hot_label = tf.reshape(tf.one_hot(orig_idx, orig_preds.shape[-1]), (1, -1))
        target_one_hot = tf.reshape(tf.one_hot(TARGET_CLASS, orig_preds.shape[-1]), (1, -1))
        
        metrics['Baseline']['correct'] += 1 # We use this prediction as the ground truth baseline
        
        # Function to evaluate an attack
        def evaluate_attack(attack_name, adv_img, targeted=False):
            adv_preds = model(adv_img, training=False)
            adv_idx = tf.argmax(adv_preds, axis=-1).numpy()[0]
            
            l2_dist = np.linalg.norm(img.numpy() - adv_img.numpy())
            metrics[attack_name]['l2'] += l2_dist
            
            if adv_idx == orig_idx:
                metrics[attack_name]['correct'] += 1 # Attack Failed, Model Resisted
            
            if targeted:
                if adv_idx == TARGET_CLASS:
                    metrics[attack_name]['success'] += 1 # Targeted Attack Succeeded
            else:
                if adv_idx != orig_idx:
                    metrics[attack_name]['success'] += 1 # Untargeted Attack Succeeded
                    
        # FGSM
        adv_fgsm = fgsm_attack(img, one_hot_label, eps, model, c_min, c_max)
        evaluate_attack('FGSM', adv_fgsm)
        
        # PGD
        adv_pgd = pgd_attack(img, one_hot_label, eps, model, c_min, c_max, iters=10)
        evaluate_attack('PGD', adv_pgd)
        
        # C&W
        adv_cw = cw_attack(img, one_hot_label, model, config['box_min'], config['box_max'], max_iters=50)
        evaluate_attack('C&W', adv_cw)

        # DeepFool
        adv_df = deepfool_attack(img, model, c_min, c_max, max_iter=20)
        evaluate_attack('DeepFool', adv_df)
        
        # Targeted I-FGSM
        adv_tifgsm = targeted_ifgsm_attack(img, target_one_hot, eps, model, c_min, c_max, iters=20)
        evaluate_attack('T-IFGSM', adv_tifgsm, targeted=True)
        
    # Aggregate and Save Results
    total = len(image_paths)
    for attack, data in metrics.items():
        results.append({
            'Model': model_name,
            'Attack': attack,
            'Accuracy (%)': (data['correct'] / total) * 100,
            'ASR (%)': (data.get('success', 0) / total) * 100,
            'Avg_L2': data['l2'] / total
        })

print("\nEvaluation Complete!")

### Export Results to CSV
We save the aggregated metrics into a DataFrame and export it. Subsequent notebooks will load this file directly.

In [ ]:
df_results = pd.DataFrame(results)

# Display the results table
display(df_results)

# Save to CSV
output_filename = 'robustness_metrics.csv'
df_results.to_csv(output_filename, index=False)
print(f"Metrics successfully saved to '{output_filename}'")